In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import re, gzip, random
from collections import OrderedDict
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Sequence, Optional
import pandas as pd

# ---------- FASTA I/O ----------
def read_fasta(path: str) -> OrderedDict:
    op = gzip.open if path.endswith(".gz") else open
    fa = OrderedDict()
    name, buff = None, []
    with op(path, "rt") as f:
        for line in f:
            if not line.strip():
                continue
            if line.startswith(">"):
                if name:
                    fa[name] = "".join(buff).upper()
                name = line[1:].strip().split()[0]; buff = []
            else:
                buff.append(line.strip())
        if name:
            fa[name] = "".join(buff).upper()
    return fa

def write_fasta(path: str, fa: Dict[str,str], width: int = 60):
    op = gzip.open if path.endswith(".gz") else open
    with op(path, "wt") as out:
        for name, seq in fa.items():
            out.write(f">{name}\n")
            for i in range(0, len(seq), width):
                out.write(seq[i:i+width] + "\n")

def subset_fasta(fa: Dict[str,str], keep_names: Sequence[str]) -> OrderedDict:
    return OrderedDict((k, v) for k, v in fa.items() if k in keep_names)

# ---------- HPV helpers ----------
RCMAP = str.maketrans("ACGTNacgtn", "TGCANtgcan")
def rc(s: str) -> str:
    return s.translate(RCMAP)[::-1]

def normalize(n: int, L: int) -> int:
    """Normalize any integer to 1..L for circular coordinates."""
    return ((n - 1) % L) + 1

def extract_circular(seq: str, a: int, b: int) -> str:
    """Extract [a,b] 1-based inclusive on a circular sequence (wrap if a>b)."""
    L = len(seq)
    a = normalize(a, L); b = normalize(b, L)
    a0, b0 = a-1, b-1
    if a <= b:
        return seq[a0:b0+1]
    else:
        return seq[a0:] + seq[:b0+1]

def build_hpv_unit(hpv_seq: str, paths: List[Tuple[int,int]], hpv_strand: str, copies: int = 1) -> str:
    segs = [extract_circular(hpv_seq, a, b) for (a,b) in paths]
    unit = "".join(segs)
    unit = rc(unit) if hpv_strand == "-" else unit
    return unit * copies

# ---------- composite insert (hpv / host / literal) ----------
def build_composite_insert(hpv_seq: str, blocks: List[Dict], host_ref: Dict[str,str]) -> Tuple[str, str]:
    """
      - {"type":"hpv", "paths":[(a,b), ...], "strand":"+"|"-", "copies":1}
      - {"type":"host", "chrom":"chr17", "start":36094082, "end":36095602, "strand":"+"|"-"}
      - {"type":"literal", "seq":"ACGT..."}
    return: (insert_seq, desc)
    """
    pieces, descs = [], []
    for blk in blocks:
        t = blk.get("type")
        if t == "hpv":
            paths   = blk["paths"]
            strand  = blk.get("strand", "+")
            copies  = int(blk.get("copies", 1))
            unit    = build_hpv_unit(hpv_seq, paths, strand, copies=copies)
            pieces.append(unit)
            pstr = ",".join([f"{a}-{b}" for a,b in paths])
            rep  = f"x{copies}" if copies != 1 else ""
            descs.append(f"HPV:{pstr}({strand}){rep}")
        elif t == "host":
            chrom = blk["chrom"]; a = int(blk["start"]); b = int(blk["end"])
            strand = blk.get("strand", "+")
            if chrom not in host_ref:
                raise ValueError(f"Host block chrom not loaded: {chrom}")
            ref = host_ref[chrom]
            if not (1 <= a <= len(ref) and 1 <= b <= len(ref)):
                raise ValueError(f"Host block coords out of range: {chrom}:{a}-{b} (len={len(ref)})")
            if a > b:
                raise ValueError(f"Host block expects linear coords a<=b, got {a}-{b}")
            sub = ref[a-1:b]
            sub = rc(sub) if strand == "-" else sub
            pieces.append(sub)
            descs.append(f"{chrom}:{a}-{b}({strand})")
        elif t == "literal":
            seq = blk["seq"].upper()
            pieces.append(seq)
            descs.append(f"LIT:{len(seq)}")
        else:
            raise ValueError(f"Unknown block type: {t}")
    return "".join(pieces), "+".join(descs)

# ---------- intervals & placement ----------
def overlaps(intervals: List[Tuple[int,int]], new_iv: Tuple[int,int]) -> bool:
    a,b = new_iv
    for x,y in intervals:
        if not (b < x or a > y):  # overlap
            return True
    return False

def choose_random_site(chrom_sizes: Dict[str,int], allow_chrs: List[str], del_len: int,
                       occupied: Dict[str,List[Tuple[int,int]]],
                       rng: random.Random, max_retry: int = 10000) -> Tuple[str,int]:
    """Length-weighted chromosome sampling; ensure deletion window fits and doesn't overlap on that chrom."""
    weights = [(c, chrom_sizes[c]) for c in allow_chrs]
    tot = sum(w for _,w in weights)
    if tot <= 0:
        raise ValueError("Total chromosome length is zero in allow_chrs.")
    for _ in range(max_retry):
        r = rng.randint(1, tot)
        acc = 0; chrom = None
        for c,w in weights:
            acc += w
            if r <= acc:
                chrom = c; break
        L = chrom_sizes[chrom]
        max_pos = max(1, L - del_len)
        pos = rng.randint(1, max_pos)
        del_iv = (pos+1, pos+del_len) if del_len>0 else (pos+1, pos)
        if not overlaps(occupied[chrom], del_iv):
            occupied[chrom].append(del_iv)
            return chrom, pos
    raise RuntimeError("Failed to place event without overlap")

# ---------- event schema ----------
@dataclass
class Event:
    event_id: str
    chrom: str
    host_pos: int       # insert AFTER this base (1-based)
    del_len: int        # delete [pos+1, pos+del_len]
    ins_len: int
    hpv_ref: str
    hpv_path: str       # description of blocks/paths
    hpv_strand: str     # '+' or '-' or '.'
    copies: int
    source: str         # e.g., list tag or EXTRA

# ---------- site parsing ----------
def parse_site(s: str) -> Tuple[str, int]:
    """Parse 'chr9:97813335' or 'chr9 97813335'."""
    s = s.strip()
    m = re.match(r'^(chr[^\s:]+)\s*[:\s]\s*(\d+)$', s)
    if not m:
        raise ValueError(f"Bad site format: {s}")
    return m.group(1), int(m.group(2))

def normalize_pos_within_chr(pos: int, L: int) -> int:
    return ((pos - 1) % L) + 1

def pick_sites_from_list(cands: List[str],
                         allow_chrs: List[str],
                         chrom_sizes: Dict[str,int],
                         occupied: Dict[str,List[Tuple[int,int]]],
                         k: int,
                         del_len: int,
                         rng: random.Random) -> List[Tuple[str,int]]:
    """Pick k valid sites from candidate list (no overlap; window fits)."""
    out: List[Tuple[str,int]] = []
    tried = set()
    order = list(cands)
    rng.shuffle(order)

    for s in order:
        if len(out) >= k:
            break
        if s in tried:
            continue
        tried.add(s)
        try:
            chrom, pos = parse_site(s)
        except Exception:
            continue
        if chrom not in allow_chrs:
            continue
        L = chrom_sizes.get(chrom)
        if not L:
            continue
        pos = normalize_pos_within_chr(pos, L)
        e0 = pos + del_len
        if e0 > L:
            continue
        del_iv = (pos+1, pos+del_len) if del_len > 0 else (pos+1, pos)
        if overlaps(occupied[chrom], del_iv):
            continue
        occupied[chrom].append(del_iv)
        out.append((chrom, pos))

    if len(out) < k:
        raise RuntimeError(f"Not enough valid sites from list (need {k}, got {len(out)}).")
    return out

# ---------- core ----------
def insert_events_from_site_lists(host_fa_path: str,
                                  hpv_fa_path: str,
                                  out_fa_path: str,
                                  site_lists: Dict[str, List[str]],
                                  k_per_list: int = 2,
                                  n_random: int = 2,
                                  allow_chrs: Tuple[str,...] = ("chr1","chr2","chr3","chr4","chr5","chr6","chr7","chr8","chr9","chr10",
                                                                "chr11","chr12","chr13","chr14","chr15","chr16","chr17","chr18","chr19",
                                                                "chr20","chr21","chr22","chrX","chrY"),
                                  recipes: Optional[List[Dict]] = None,
                                  out_prefix: str = "run_sitelist",
                                  seed: int = 20251004,
                                  extra_sites: Optional[List[str]] = None,
                                  output_only_modified: bool = True):
    """
    每个 list 取 k_per_list 个 + 从 extra_sites 取 n_random 个 → 逐点应用 recipes 轮换产生事件。
    """
    rng = random.Random(seed)

    # load & subset host / hpv
    host = read_fasta(host_fa_path)
    host = subset_fasta(host, [c for c in allow_chrs if c in host])
    if not host:
        raise ValueError("No allowed chromosomes found in host FASTA.")
    hpv = read_fasta(hpv_fa_path)
    if len(hpv) != 1:
        raise ValueError("HPV FASTA should contain exactly one sequence.")
    hpv_name, hpv_seq = list(hpv.items())[0]

    chrom_sizes = {c: len(s) for c,s in host.items()}
    mutable = OrderedDict((c, list(seq)) for c,seq in host.items())
    occupied = {c: [] for c in host.keys()}

    # default recipes if None
    if recipes is None:
        recipes = [
            {"name":"C", "paths":[(1471,7906),(1,2727)], "strand": "+", "del_len": 33063, "copies": 1},
            {"name":"D", "paths":[(5983,7906),(1,7906),(1,7906),(1,3406)], "strand": "+", "del_len": 342, "copies": 1},
        ]

    def recipe_max_del(recs: List[Dict]) -> int:
        return max(int(r.get("del_len", 0)) for r in recs) if recs else 0

    max_dlen = recipe_max_del(recipes)

    # choose sites from each list
    chosen_sites: List[Tuple[str,int,str]] = []
    for tag, lst in site_lists.items():
        picks = pick_sites_from_list(lst, list(allow_chrs), chrom_sizes, occupied,
                                     k=k_per_list, del_len=max_dlen, rng=rng)
        for chrom, pos in picks:
            chosen_sites.append((chrom, pos, tag))

    # add extra sites
    if n_random > 0:
        if not extra_sites:
            raise ValueError("n_random > 0 but extra_sites is None or empty.")
        picks = pick_sites_from_list(extra_sites, list(allow_chrs), chrom_sizes, occupied,
                                     k=n_random, del_len=max_dlen, rng=rng)
        for i,(chrom,pos) in enumerate(picks, start=1):
            chosen_sites.append((chrom, pos, f"EXTRA{i}"))

    # build & apply events
    events: List[Event] = []
    bedpe_rows = []
    recipe_idx = 0
    modified_chroms = set()

    for idx, (chrom, pos, src) in enumerate(chosen_sites, start=1):
        r = recipes[recipe_idx % len(recipes)]
        recipe_idx += 1

        # build insert sequence from recipe
        if "blocks" in r:
            insert_seq, hpv_path_desc = build_composite_insert(hpv_seq, r["blocks"], host_ref=host)
            dlen   = int(r.get("del_len", 0))
            copies = 1
            strand = "."
        else:
            paths   = r["paths"]
            strand  = r.get("strand", "+")
            dlen    = int(r.get("del_len", 0))
            copies  = int(r.get("copies", 1))
            insert_seq = build_hpv_unit(hpv_seq, paths, strand, copies=copies)
            hpv_path_desc = ";".join([f"{a}-{b}" for a,b in paths]) + f"({strand})"

        ins_len = len(insert_seq)

        # ensure deletion window fits
        L = chrom_sizes[chrom]
        if pos + dlen > L:
            raise RuntimeError(f"Chosen site {chrom}:{pos} with del_len={dlen} exceeds chromosome length {L}.")

        # mutate host
        seq = mutable[chrom]
        s0 = pos
        e0 = pos + dlen
        new_seq = seq[:s0] + list(insert_seq) + seq[e0:]
        mutable[chrom] = new_seq
        modified_chroms.add(chrom)

        ev_id = f"E{idx}_{src}_{r.get('name', f'R{recipe_idx}')}"
        events.append(Event(
            event_id=ev_id,
            chrom=chrom,
            host_pos=pos,
            del_len=dlen,
            ins_len=ins_len,
            hpv_ref=hpv_name,
            hpv_path=hpv_path_desc,
            hpv_strand=strand,
            copies=copies,
            source=src
        ))
        bedpe_rows.append({
            "chrom1": chrom, "start1": pos, "end1": pos+1,
            "chrom2": hpv_name, "start2": 1, "end2": 2,
            "name": ev_id, "score": ".",
            "strand1": "+", "strand2": strand
        })

    # write mutated FASTA
    if output_only_modified:
        to_write = [(c, "".join(mutable[c])) for c in mutable.keys() if c in modified_chroms]
        if not to_write:
            raise RuntimeError("No chromosomes had insertions; nothing to write. "
                               "Disable output_only_modified or check site selection.")
    else:
        to_write = [(c, "".join(seq)) for c,seq in mutable.items()]

    mutated = OrderedDict((f"{c}|HPVsiteLists", seq) for c, seq in to_write)
    write_fasta(out_fa_path, mutated)

    # truth tables
    ev_df = pd.DataFrame([asdict(e) for e in events])
    ev_path = out_prefix + ".events.tsv"
    ev_df.to_csv(ev_path, sep="\t", index=False)

    bedpe_df = pd.DataFrame(bedpe_rows)
    bedpe_path = out_prefix + ".bedpe.tsv"
    bedpe_df.to_csv(bedpe_path, sep="\t", index=False)

    print(f"FASTA written: {out_fa_path}")
    print(f"Events table : {ev_path}")
    print(f"BEDPE table  : {bedpe_path}")
    return ev_path, bedpe_path, out_fa_path


# ===================== Example usage =====================
if __name__ == "__main__":
    LINE = [
        "chr11:131789342","chr1:16783209","chr18:22565190","chr2:151136063",
        "chr3:81992132","chr3:82226566","chr7:83120494","chr9:26645008",
        "chr9:97813335","chr9:97814224","chr9:97902954","chr9:97908821",
        "chr9:97914603","chr9:97944420","chr9:97945714","chr9:97952091"
    ]
    SINE = [
        "chr12:66057682","chr3:189879690","chr6:36910796","chr6:37167442",
        "chr6:37176440","chr6:79484344","chr9:116921587"
    ]
    Dup = ["chr1:16783209", "chr1:234783386"]
    lowgc = ["chr4:103351588","chr9:99956760"]

  
    extra_sites = [
        "chr2:32916236","chr3:141401612","chr3:189889702","chr3:189895061","chr3:81785783",
        "chr3:93470597","chr3:93470710","chr6:37161250","chr6:37175775","chr8:46619611",
        "chr9:97833766","chr9:97857784","chr9:97860564","chr9:97869989","chr9:97891262",
        "chr9:97899374","chr9:97901339","chr9:97940224","chr9:97942608","chr9:97942781",
    ]

    site_lists = {"LINE": LINE, "SINE": SINE, "Dup": Dup, "lowgc": lowgc}


    allow = ("chr1","chr2","chr3","chr4","chr6","chr7","chr8","chr9","chr11","chr12","chr17","chr18")

    # ---------- Recipes: A/B/C/D/E/F ----------
    recipes = [
        # # A
        # {"name":"A",
        #  "paths":[(1706,6292)],
        #  "strand": "+",
        #  "del_len": 0,
        #  "copies": 1},

        # # B
        # {"name":"B",
        #  "paths":[(5305,7906),(1,3372)],
        #  "strand": "-",
        #  "del_len": 134_237,
        #  "copies": 1},

        # # C
        # {"name":"C",
        #  "paths":[(1471,7906),(1,2727)],
        #  "strand": "+",
        #  "del_len": 33063,
        #  "copies": 1},

        # # D
        # {"name":"D",
        #  "paths":[(5983,7906),(1,7906),(1,7906),(1,3406)],
        #  "strand": "+",
        #  "del_len": 342,
        #  "copies": 1},

        # E = E_new1_rep3 : HPV + host(chr17) + HPV + host(chr17) + HPV
        {"name":"E",
         "blocks":[
            {"type":"hpv","paths":[(2599,4572)],"strand":"+","copies":1},
            {"type":"host","chrom":"chr17","start":36094082,"end":36095602,"strand":"+"},
            {"type":"hpv","paths":[(5979,7906),(1,4572)],"strand":"+","copies":1},
            {"type":"host","chrom":"chr17","start":36094082,"end":36095602,"strand":"+"},
            {"type":"hpv","paths":[(5979,7906),(1,4572)],"strand":"+","copies":1},
         ],
         "del_len":0},

        # F = E_new2_rep1 : HPV(-) + "TATTA" + HPV(+), chr_del=651
        {"name":"F",
         "blocks":[
            {"type":"hpv","paths":[(3572,1),(7906,4100)],"strand":"-","copies":1},
            {"type":"literal","seq":"TATTA"},
            {"type":"hpv","paths":[(6000,7906),(1,2114)],"strand":"+","copies":1},
         ],
         "del_len":651},
    ]


    host_fa_path = "hg38.subset2.fa"   # 需包含 allow 中的染色体（含 chr17）
    hpv_fa_path  = "HPV16.fa"
    out_fa_path  = "Level4.withHPV.fa.gz"

    _ = insert_events_from_site_lists(
        host_fa_path=host_fa_path,
        hpv_fa_path=hpv_fa_path,
        out_fa_path=out_fa_path,
        site_lists=site_lists,
        extra_sites=extra_sites,  
        k_per_list=2,
        n_random=92,
        allow_chrs=allow,
        recipes=recipes,          
        out_prefix="run_planEF",
        seed=20251004,
        output_only_modified=True
    )


FASTA written: Level4.withHPV.fa.gz
Events table : run_planEF.events.tsv
BEDPE table  : run_planEF.bedpe.tsv


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import re, gzip, random
from collections import OrderedDict
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Sequence, Optional
import pandas as pd

# ---------- FASTA I/O ----------
def read_fasta(path: str) -> OrderedDict:
    op = gzip.open if path.endswith(".gz") else open
    fa = OrderedDict()
    name, buff = None, []
    with op(path, "rt") as f:
        for line in f:
            if not line.strip():
                continue
            if line.startswith(">"):
                if name:
                    fa[name] = "".join(buff).upper()
                name = line[1:].strip().split()[0]; buff = []
            else:
                buff.append(line.strip())
        if name:
            fa[name] = "".join(buff).upper()
    return fa

def write_fasta(path: str, fa: Dict[str,str], width: int = 60):
    op = gzip.open if path.endswith(".gz") else open
    with op(path, "wt") as out:
        for name, seq in fa.items():
            out.write(f">{name}\n")
            for i in range(0, len(seq), width):
                out.write(seq[i:i+width] + "\n")

def subset_fasta(fa: Dict[str,str], keep_names: Sequence[str]) -> OrderedDict:
    return OrderedDict((k, v) for k, v in fa.items() if k in keep_names)

# ---------- HPV helpers ----------
RCMAP = str.maketrans("ACGTNacgtn", "TGCANtgcan")
def rc(s: str) -> str:
    return s.translate(RCMAP)[::-1]

def normalize(n: int, L: int) -> int:
    """Normalize any integer to 1..L for circular coordinates."""
    return ((n - 1) % L) + 1

def extract_circular(seq: str, a: int, b: int) -> str:
    """Extract [a,b] 1-based inclusive on a circular sequence (wrap if a>b)."""
    L = len(seq)
    a = normalize(a, L); b = normalize(b, L)
    a0, b0 = a-1, b-1
    if a <= b:
        return seq[a0:b0+1]
    else:
        return seq[a0:] + seq[:b0+1]

def build_hpv_unit(hpv_seq: str, paths: List[Tuple[int,int]], hpv_strand: str, copies: int = 1) -> str:
    segs = [extract_circular(hpv_seq, a, b) for (a,b) in paths]
    unit = "".join(segs)
    unit = rc(unit) if hpv_strand == "-" else unit
    return unit * copies

# ---------- composite insert (hpv / host / literal) ----------
def build_composite_insert(hpv_seq: str, blocks: List[Dict], host_ref: Dict[str,str]) -> Tuple[str, str]:
    pieces, descs = [], []
    for blk in blocks:
        t = blk.get("type")
        if t == "hpv":
            paths   = blk["paths"]
            strand  = blk.get("strand", "+")
            copies  = int(blk.get("copies", 1))
            unit    = build_hpv_unit(hpv_seq, paths, strand, copies=copies)
            pieces.append(unit)
            pstr = ",".join([f"{a}-{b}" for a,b in paths])
            rep  = f"x{copies}" if copies != 1 else ""
            descs.append(f"HPV:{pstr}({strand}){rep}")
        elif t == "host":
            chrom = blk["chrom"]; a = int(blk["start"]); b = int(blk["end"])
            strand = blk.get("strand", "+")
            if chrom not in host_ref:
                raise ValueError(f"Host block chrom not loaded: {chrom}")
            ref = host_ref[chrom]
            if not (1 <= a <= len(ref) and 1 <= b <= len(ref)):
                raise ValueError(f"Host block coords out of range: {chrom}:{a}-{b} (len={len(ref)})")
            if a > b:
                raise ValueError(f"Host block expects linear coords a<=b, got {a}-{b}")
            sub = ref[a-1:b]
            sub = rc(sub) if strand == "-" else sub
            pieces.append(sub)
            descs.append(f"{chrom}:{a}-{b}({strand})")
        elif t == "literal":
            seq = blk["seq"].upper()
            pieces.append(seq)
            descs.append(f"LIT:{len(seq)}")
        else:
            raise ValueError(f"Unknown block type: {t}")
    return "".join(pieces), "+".join(descs)

# ---------- intervals & placement ----------
def overlaps(intervals: List[Tuple[int,int]], new_iv: Tuple[int,int]) -> bool:
    a,b = new_iv
    for x,y in intervals:
        if not (b < x or a > y):  # overlap
            return True
    return False

def choose_random_site(chrom_sizes: Dict[str,int], allow_chrs: List[str], del_len: int,
                       occupied: Dict[str,List[Tuple[int,int]]],
                       rng: random.Random, max_retry: int = 10000) -> Tuple[str,int]:
    """Length-weighted chromosome sampling; ensure deletion window fits and doesn't overlap on that chrom."""
    weights = [(c, chrom_sizes[c]) for c in allow_chrs]
    tot = sum(w for _,w in weights)
    if tot <= 0:
        raise ValueError("Total chromosome length is zero in allow_chrs.")
    for _ in range(max_retry):
        r = rng.randint(1, tot)
        acc = 0; chrom = None
        for c,w in weights:
            acc += w
            if r <= acc:
                chrom = c; break
        L = chrom_sizes[chrom]
        max_pos = max(1, L - del_len)
        pos = rng.randint(1, max_pos)
        # del_iv = (pos+1, pos+del_len) if del_len>0 else (pos+1, pos)
        core_start = pos + 1
        core_end   = pos + del_len if del_len > 0 else pos

        # expand window by min_gap to enforce spacing
        start = max(1, core_start - 50000)
        end   = min(L, core_end + 50000)

        del_iv = (start, end)
        if not overlaps(occupied[chrom], del_iv):
            occupied[chrom].append(del_iv)
            return chrom, pos
    raise RuntimeError("Failed to place event without overlap")

# ---------- event schema ----------
@dataclass
class Event:
    event_id: str
    chrom: str
    host_pos: int
    del_len: int
    ins_len: int
    hpv_ref: str
    hpv_path: str
    hpv_strand: str
    copies: int
    source: str

# ---------- site parsing ----------
def parse_site(s: str) -> Tuple[str, int]:
    s = s.strip()
    m = re.match(r'^(chr[^\s:]+)\s*[:\s]\s*(\d+)$', s)
    if not m:
        raise ValueError(f"Bad site format: {s}")
    return m.group(1), int(m.group(2))

def normalize_pos_within_chr(pos: int, L: int) -> int:
    return ((pos - 1) % L) + 1

def pick_sites_from_list(cands: List[str],
                         allow_chrs: List[str],
                         chrom_sizes: Dict[str,int],
                         occupied: Dict[str,List[Tuple[int,int]]],
                         k: int,
                         del_len: int,
                         rng: random.Random) -> List[Tuple[str,int]]:
    """
    """
    out: List[Tuple[str,int]] = []
    tried = set()
    order = list(cands)
    rng.shuffle(order)

    for s in order:
        if len(out) >= k:
            break
        if s in tried:
            continue
        tried.add(s)
        try:
            chrom, pos = parse_site(s)
        except Exception:
            continue
        if chrom not in allow_chrs:
            continue
        L = chrom_sizes.get(chrom)
        if not L:
            continue
        pos = normalize_pos_within_chr(pos, L)
        if pos + del_len > L:
            continue
        del_iv = (pos+1, pos+del_len) if del_len > 0 else (pos+1, pos)
        if overlaps(occupied[chrom], del_iv):
            continue
        occupied[chrom].append(del_iv)
        out.append((chrom, pos))

    return out

# ---------- core ----------
def insert_events_from_site_lists(host_fa_path: str,
                                  hpv_fa_path: str,
                                  out_fa_path: str,
                                  site_lists: Dict[str, List[str]],
                                  allow_chrs: Tuple[str,...],
                                  recipes: Optional[List[Dict]] = None,
                                  out_prefix: str = "run_sitelist",
                                  seed: int = 20251004,
                                  extra_sites: Optional[List[str]] = None,
                                  output_only_modified: bool = True,
                                  target_total: int = 100):
    """
    """
    rng = random.Random(seed)

    # load & subset host / hpv
    host = read_fasta(host_fa_path)
    host = subset_fasta(host, [c for c in allow_chrs if c in host])
    if not host:
        raise ValueError("No allowed chromosomes found in host FASTA.")
    hpv = read_fasta(hpv_fa_path)
    if len(hpv) != 1:
        raise ValueError("HPV FASTA should contain exactly one sequence.")
    hpv_name, hpv_seq = list(hpv.items())[0]

    chrom_sizes = {c: len(s) for c,s in host.items()}
    mutable = OrderedDict((c, list(seq)) for c,seq in host.items())
    occupied = {c: [] for c in host.keys()}

    # default recipes
    if recipes is None:
        recipes = [
            {"name":"C", "paths":[(1471,7906),(1,2727)], "strand": "+", "del_len": 33063, "copies": 1},
            {"name":"D", "paths":[(5983,7906),(1,7906),(1,7906),(1,3406)], "strand": "+", "del_len": 342, "copies": 1},
        ]

    def recipe_max_del(recs: List[Dict]) -> int:
        return max(int(r.get("del_len", 0)) for r in recs) if recs else 0

    max_dlen = recipe_max_del(recipes)

    chosen_sites: List[Tuple[str,int,str]] = []

    # 1) fixed point
    for tag, lst in site_lists.items():
        if not lst:
            continue
        picks = pick_sites_from_list(
            lst,
            list(allow_chrs),
            chrom_sizes,
            occupied,
            k=len(lst),          # all
            del_len=max_dlen,
            rng=rng
        )
        for chrom, pos in picks:
            chosen_sites.append((chrom, pos, tag))

    # 2) extra_sites try all in
    if extra_sites:
        extra_picks = pick_sites_from_list(
            extra_sites,
            list(allow_chrs),
            chrom_sizes,
            occupied,
            k=len(extra_sites),
            del_len=max_dlen,
            rng=rng
        )
        for chrom, pos in extra_picks:
            chosen_sites.append((chrom, pos, "EXTRA"))

    # 3) random sites
    while len(chosen_sites) < target_total:
        chrom, pos = choose_random_site(
            chrom_sizes=chrom_sizes,
            allow_chrs=list(allow_chrs),
            del_len=max_dlen,
            occupied=occupied,
            rng=rng
        )
        chosen_sites.append((chrom, pos, f"RAND{len(chosen_sites)+1}"))

    # 4) build & apply events
    events: List[Event] = []
    bedpe_rows = []
    recipe_idx = 0
    modified_chroms = set()
    #  chr SITES APPLY FROM LEFT TO RIGHT，keep pos always“ORIGINAL reference POSITION”
    chr_order = {c:i for i, c in enumerate(allow_chrs)}  # keeo allow_chrs sequence
    chosen_sites = sorted(
        chosen_sites,
        key=lambda x: (chr_order.get(x[0], 10**9), -x[1])  # chrom follow allow_chrs, pos decrease
    )


    for idx, (chrom, pos, src) in enumerate(chosen_sites, start=1):
        r = recipes[recipe_idx % len(recipes)]
        recipe_idx += 1

        if "blocks" in r:
            insert_seq, hpv_path_desc = build_composite_insert(hpv_seq, r["blocks"], host_ref=host)
            dlen   = int(r.get("del_len", 0))
            copies = 1
            strand = "."
        else:
            paths   = r["paths"]
            strand  = r.get("strand", "+")
            dlen    = int(r.get("del_len", 0))
            copies  = int(r.get("copies", 1))
            insert_seq = build_hpv_unit(hpv_seq, paths, strand, copies=copies)
            hpv_path_desc = ";".join([f"{a}-{b}" for a,b in paths]) + f"({strand})"

        ins_len = len(insert_seq)
        L = chrom_sizes[chrom]
        if pos + dlen > L:
            raise RuntimeError(f"Chosen site {chrom}:{pos} with del_len={dlen} exceeds chromosome length {L}.")

        seq = mutable[chrom]
        s0 = pos
        e0 = pos + dlen
        new_seq = seq[:s0] + list(insert_seq) + seq[e0:]
        mutable[chrom] = new_seq
        modified_chroms.add(chrom)

        ev_id = f"E{idx}_{src}_{r.get('name', f'R{recipe_idx}')}"
        events.append(Event(
            event_id=ev_id,
            chrom=chrom,
            host_pos=pos,
            del_len=dlen,
            ins_len=ins_len,
            hpv_ref=hpv_name,
            hpv_path=hpv_path_desc,
            hpv_strand=strand,
            copies=copies,
            source=src
        ))
        bedpe_rows.append({
            "chrom1": chrom, "start1": pos, "end1": pos+1,
            "chrom2": hpv_name, "start2": 1, "end2": 2,
            "name": ev_id, "score": ".",
            "strand1": "+", "strand2": strand
        })

    # write mutated FASTA
    if output_only_modified:
        to_write = [(c, "".join(mutable[c])) for c in mutable.keys() if c in modified_chroms]
        if not to_write:
            raise RuntimeError("No chromosomes had insertions; nothing to write.")
    else:
        to_write = [(c, "".join(seq)) for c,seq in mutable.items()]

    mutated = OrderedDict((f"{c}|HPVsiteLists", seq) for c, seq in to_write)
    write_fasta(out_fa_path, mutated)

    ev_df = pd.DataFrame([asdict(e) for e in events])
    ev_path = out_prefix + ".events.tsv"
    ev_df.to_csv(ev_path, sep="\t", index=False)

    bedpe_df = pd.DataFrame(bedpe_rows)
    bedpe_path = out_prefix + ".bedpe.tsv"
    bedpe_df.to_csv(bedpe_path, sep="\t", index=False)

    print(f"FASTA written: {out_fa_path}")
    print(f"Events table : {ev_path}")
    print(f"BEDPE table  : {bedpe_path}")
    return ev_path, bedpe_path, out_fa_path


# ===================== Example usage =====================
if __name__ == "__main__":
    LINE = [
        "chr11:131789342","chr1:16783209","chr18:22565190","chr2:151136063",
        "chr3:81992132","chr3:82226566","chr7:83120494","chr9:26645008",
        "chr9:97813335","chr9:97814224","chr9:97902954","chr9:97908821",
        "chr9:97914603","chr9:97944420","chr9:97945714","chr9:97952091"
    ]
    SINE = [
        "chr12:66057682","chr3:189879690","chr6:36910796","chr6:37167442",
        "chr6:37176440","chr6:79484344","chr9:116921587"
    ]
    Dup = ["chr1:16783209", "chr1:234783386"]
    lowgc = ["chr4:103351588","chr9:99956760"]

    extra_sites = [
        "chr2:32916236","chr3:141401612","chr3:189889702","chr3:189895061","chr3:81785783",
        "chr3:93470597","chr6:37161250","chr6:37175775",
        "chr9:97833766","chr9:97857784","chr9:97860564","chr9:97869989","chr9:97891262",
        "chr9:97899374","chr9:97901339","chr9:97940224","chr9:97942608","chr9:97942781",
        "chrX:96960123","chrX:96983771","chrX:97114880","chrX:97125896",
    ]

    site_lists = {"LINE": LINE, "SINE": SINE, "Dup": Dup, "lowgc": lowgc}

    allow = ("chr1","chr2","chr3","chr4","chr6","chr7","chr9","chr11","chr12","chr17","chr18","chrX")

    recipes = [
              # {
              #       "name": "A",
              #       "blocks": [
              #           {
              #               "type": "hpv",
              #               "paths": [(1706, 6292)],
              #               "strand": "+",
              #               "copies": 1,
              #           }
              #       ],
              #       "del_len": 0,
              #   },
            
              #   # B: HPV:(5305-7906, 1-3372)(-), del_len=134237
              #   {
              #       "name": "B",
              #       "blocks": [
              #           {
              #               "type": "hpv",
              #               "paths": [(5305, 7906), (1, 3372)],
              #               "strand": "-",
              #               "copies": 1,
              #           }
              #       ],
              #       "del_len": 134_237,
              #   },
            
                # C: HPV:(1471-7906, 1-2727)(+), del_len=33063
                # {
                #     "name": "C",
                #     "blocks": [
                #         {
                #             "type": "hpv",
                #             "paths": [(1471, 7906), (1, 2727)],
                #             "strand": "+",
                #             "copies": 1,
                #         }
                #     ],
                #     "del_len": 33_063,
                # },
            
                # # D: HPV:(5983-7906, 1-7906, 1-7906, 1-3406)(+), del_len=342
                # {
                #     "name": "D",
                #     "blocks": [
                #         {
                #             "type": "hpv",
                #             "paths": [(5983, 7906), (1, 7906), (1, 7906), (1, 3406)],
                #             "strand": "+",
                #             "copies": 1,
                #         }
                #     ],
                #     "del_len": 342,
                # },
        {"name":"E",
         "blocks":[
            {"type":"hpv","paths":[(2599,4572)],"strand":"+","copies":1},
            {"type":"host","chrom":"chr17","start":36094082,"end":36099602,"strand":"+"},
            {"type":"hpv","paths":[(5979,7906),(1,4572)],"strand":"+","copies":1},
            {"type":"host","chrom":"chr17","start":36094082,"end":36099602,"strand":"+"},
            {"type":"hpv","paths":[(5979,7906),(1,4572)],"strand":"+","copies":1},
         ],
         "del_len":0},
        {"name":"F",
         "blocks":[
            {"type":"hpv","paths":[(4100,7906),(1,3572)],"strand":"-","copies":1},
            {"type":"literal","seq":"TATTA"},
            {"type":"hpv","paths":[(6000,7906),(1,2114)],"strand":"+","copies":1},
         ],
         "del_len":651},
        {"name":"UD2",
         "blocks":[
            {"type":"hpv","paths":[(1,3106)],"strand":"+","copies":1},
            {"type":"host","chrom":"chrX","start":97114879,"end":97125896,"strand":"+"},
            {"type":"hpv","paths":[(7710,7892)],"strand":"+","copies":1},
            {"type":"hpv","paths":[(1,3106)],"strand":"+","copies":1},
            {"type":"host","chrom":"chrX","start":97114879,"end":97125896,"strand":"+"},
            {"type":"hpv","paths":[(7710,7892)],"strand":"+","copies":1},
            {"type":"hpv","paths":[(1,3106)],"strand":"+","copies":1},
            {"type":"host","chrom":"chrX","start":97114879,"end":97125896,"strand":"+"},
            {"type":"hpv","paths":[(7710,7892)],"strand":"+","copies":1},
            {"type":"hpv","paths":[(1,3106)],"strand":"+","copies":1},
            {"type":"host","chrom":"chrX","start":97114879,"end":97125896,"strand":"+"},
            {"type":"hpv","paths":[(7710,7892)],"strand":"+","copies":1},
            {"type":"hpv","paths":[(1,3106)],"strand":"+","copies":1},
            {"type":"host","chrom":"chrX","start":97114879,"end":97125896,"strand":"+"},
            {"type":"hpv","paths":[(7710,7892)],"strand":"+","copies":1},
            {"type":"hpv","paths":[(1,3106)],"strand":"+","copies":1},
            {"type":"host","chrom":"chrX","start":97114879,"end":97125896,"strand":"+"},
            {"type":"hpv","paths":[(7710,7892)],"strand":"+","copies":1}, 
         ],
         "del_len":0},
    ]

    host_fa_path = "hg38.p14.fa"
    hpv_fa_path  = "HPV16.fa"
    out_fa_path  = "Level3.withHPV.fa.gz"

    _ = insert_events_from_site_lists(
        host_fa_path=host_fa_path,
        hpv_fa_path=hpv_fa_path,
        out_fa_path=out_fa_path,
        site_lists=site_lists,
        extra_sites=extra_sites,
        allow_chrs=allow,
        recipes=recipes,
        out_prefix="100_1218planEFG",
        seed=20251110,
        output_only_modified=True,
        target_total=100,   
    )


FASTA written: Level3.withHPV.fa.gz
Events table : 100_1218planEFG.events.tsv
BEDPE table  : 100_1218planEFG.bedpe.tsv
